# 10 — Expectation Values

Make precise the single number a real experiment reports — the expectation value $\langle A\rangle = \langle\psi|A|\psi\rangle$ — and build the bridge to tomography.

**Prerequisites:** `01/07_pauli_gates`, `01/04_measurement_and_born_rule`.

**What you'll be able to do**
- Compute an expectation value $\langle A\rangle = \langle\psi|A|\psi\rangle$ and read $\langle Z\rangle = P(0) - P(1)$.
- Estimate $\langle Z\rangle$ from a finite number of measurements and watch it converge.
- See that $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ are exactly what `01/15` uses to reconstruct a state.

A single measurement returns a random $0$ or $1$. Repeat it many times and *average*, and you get a stable, meaningful number — the expectation value. It is what experiments actually report, and the three Pauli expectations are enough to pin down a whole qubit. This notebook turns the duality you met in `01/07` into a measurement recipe.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_PLUS, qubit_state, born_probabilities, sample_counts
from qot_course.quantum.gates import PAULI_X, PAULI_Y, PAULI_Z, HADAMARD, apply_gate, expectation

np.random.seed(42)
viz.use_course_style()

## 1. The expectation value, and what $\langle Z\rangle$ means

For an observable $A$ (a Hermitian operator) and a state $|\psi\rangle$, the **expectation value** is

$$ \langle A\rangle = \langle\psi|A|\psi\rangle. $$

It is the average outcome you would get by measuring $A$ many times. For $Z$ — whose outcomes are $+1$ on $|0\rangle$ and $-1$ on $|1\rangle$ — this average is exactly the Born-rule difference from `01/04`:

$$ \langle Z\rangle = (+1)\,P(0) + (-1)\,P(1) = P(0) - P(1). $$

In [ ]:
psi = qubit_state(theta=np.pi / 3, phi=np.pi / 5)
p = born_probabilities(psi)
print("<Z> from <psi|Z|psi>  :", round(expectation(psi, PAULI_Z), 4))
print("P(0) - P(1)           :", round(p["0"] - p["1"], 4), " (identical)")
print("<X>, <Y>, <Z>         :", np.round([expectation(psi, P) for P in (PAULI_X, PAULI_Y, PAULI_Z)], 4))

**Read the output.** The operator average $\langle\psi|Z|\psi\rangle$ equals $P(0) - P(1)$ to the digit — the expectation value *is* the weighted average of the $\pm 1$ outcomes. The three Pauli expectations together are the Bloch vector from `01/07`: a complete fingerprint of the state.

## 2. Estimating $\langle Z\rangle$ from finite measurements

A real device cannot hand you $\langle Z\rangle$ directly — it hands you outcomes. You *estimate* the expectation by averaging the $\pm 1$ results over $N$ shots:

$$ \widehat{\langle Z\rangle} = \frac{n_0 - n_1}{N}. $$

Like any average of random draws, this estimate wobbles by about $1/\sqrt{N}$ around the true value (the shot noise of `01/04`, revisited in `01/11`). Watch it settle as the shot count grows.

In [ ]:
exact = expectation(psi, PAULI_Z)
shot_counts = [16, 64, 256, 1024, 4096, 16384, 65536]
estimates = []
for i, N in enumerate(shot_counts):
    counts = sample_counts(psi, shots=N, seed=i)
    estimates.append((counts["0"] - counts["1"]) / N)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.axhline(exact, color=COLORS["highlight"], lw=2, label=f"exact <Z> = {exact:.3f}")
guide = 1 / np.sqrt(np.array(shot_counts))
ax.fill_between(shot_counts, exact - guide, exact + guide, color=COLORS["quantum"], alpha=0.18, label=r"$\pm 1/\sqrt{N}$ band")
ax.plot(shot_counts, estimates, "o-", color=COLORS["quantum"], lw=2, label="estimate")
ax.set_xscale("log")
ax.set_xlabel("number of shots N")
ax.set_ylabel(r"$\widehat{\langle Z\rangle}$")
ax.set_title("The estimate converges to the true expectation", pad=12)
ax.legend()
plt.show()

**Read the figure.** With few shots the estimate scatters widely around the true $\langle Z\rangle$; as $N$ grows it tightens into the $\pm 1/\sqrt{N}$ band and homes in on the exact value (rose line). This is the price and the promise of real measurement: more shots buy more precision, at a square-root rate. Every expectation value an experiment quotes carries an error bar of this size.

## 3. The bridge to tomography

A qubit's state is fixed by its three Pauli expectations. The density matrix `01/12` introduces is reconstructed from exactly these numbers:

$$ \rho = \tfrac{1}{2}\big(I + \langle X\rangle\,X + \langle Y\rangle\,Y + \langle Z\rangle\,Z\big). $$

You already know how to get all three: $\langle Z\rangle$ from a direct measurement, and $\langle X\rangle, \langle Y\rangle$ by first rotating those bases onto $Z$ (the Hadamard / `sdg` trick of `01/08`). That is the entire recipe of the tomography notebook `01/15` — estimate three averages, assemble the state.

In [ ]:
# <X> by measuring in the X basis: rotate with H, then average the +-1 outcomes.
N = 20000
counts_x = sample_counts(apply_gate(HADAMARD, psi), shots=N, seed=7)
x_estimate = (counts_x["0"] - counts_x["1"]) / N
print("<X> estimated (X-basis sampling):", round(x_estimate, 3))
print("<X> exact                       :", round(expectation(psi, PAULI_X), 3))

**Read the output.** Rotating with $H$ and averaging the outcomes recovers $\langle X\rangle$ to within shot noise — no new machinery, only the basis change from `01/08`. With $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ in hand, the reconstruction formula above hands you the full state. You have assembled tomography from first principles.

## Your turn

1. **A balanced state.** Compute $\langle Z\rangle$ for $|+\rangle$ and explain why it is $0$. What are $\langle X\rangle$ and $\langle Y\rangle$ for $|+\rangle$?
2. **Estimate $\langle Y\rangle$.** Using the `sdg`+`h` rotation from `01/08`, estimate $\langle Y\rangle$ of $|+i\rangle$ from samples and compare to the exact value $1$.
3. **Reconstruct.** Plug your three estimated expectations into $\rho = \tfrac12(I + \langle X\rangle X + \langle Y\rangle Y + \langle Z\rangle Z)$ and confirm it is close to $|\psi\rangle\langle\psi|$. (You have now previewed `01/15`.)

## What you built

- You defined the expectation value $\langle A\rangle = \langle\psi|A|\psi\rangle$ and saw $\langle Z\rangle = P(0) - P(1)$.
- You estimated $\langle Z\rangle$ from finite shots and watched it converge inside a $\pm 1/\sqrt{N}$ band.
- You connected $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ to the reconstruction formula at the heart of tomography (`01/15`).

The abstract bracket is now a measurement you can run and an error bar you can size. This is how quantum theory meets the lab bench — and you bridged them yourself.

## What's next

You have completed the operational toolkit for one qubit. In `01/11_shot_noise_and_first_circuit` we run it as a genuine Qiskit circuit — and the `qc.h(0)` that opens it is now an old friend, not a mystery.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 2.2.5 (observables and expectation), Cambridge University Press (2010).
- J. J. Sakurai & J. Napolitano, *Modern Quantum Mechanics*, ch. 1.4 (expectation values), 3rd ed., Cambridge University Press (2020).

**Previous:** `notebooks/01_QuantumFoundations/09_phase_and_rotation_gates.ipynb` · **Next:** `notebooks/01_QuantumFoundations/11_shot_noise_and_first_circuit.ipynb`